In [1]:
# Clona la repo posizionandoti direttamente sul tuo branch
!git clone -b Step-VIsualizer-+-Attention-Maps https://github.com/Salvatorevil03/ControlNet2026.git

# Entra nella cartella
%cd ControlNet2026

Cloning into 'ControlNet2026'...
remote: Enumerating objects: 1470, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 1470 (delta 46), reused 28 (delta 28), pack-reused 1410 (from 2)
Receiving objects: 100% (1470/1470), 122.45 MiB | 14.29 MiB/s, done.
Resolving deltas: 100% (665/665), done.
/content/ControlNet2026


In [2]:
%%capture
!pip install -q gradio pytorch-lightning omegaconf einops test-tube streamlit streamlit-drawable-canvas webdataset kornia open_clip_torch invisible-watermark torchmetrics timm addict yapf basicsr imageio-ffmpeg opencv-contrib-python

In [3]:
# @title ⬇️ Selezione Modelli ControlNet 1.5
# @markdown Spunta i modelli che desideri scaricare (attenzione: ogni modello pesa ~5.7 GB).

import os

# --- INIZIO FORM COLAB ---
Canny = True # @param {type:"boolean"}
Depth = False # @param {type:"boolean"}
HED = False # @param {type:"boolean"}
MLSD = False # @param {type:"boolean"}
Normal = False # @param {type:"boolean"}
OpenPose = False # @param {type:"boolean"}
Scribble = False # @param {type:"boolean"}
Segmentation = False # @param {type:"boolean"}
# --- FINE FORM COLAB ---

# Creiamo la lista dei modelli da scaricare in base alle tue spunte
models_to_download = []
if Canny: models_to_download.append("control_sd15_canny.pth")
if Depth: models_to_download.append("control_sd15_depth.pth")
if HED: models_to_download.append("control_sd15_hed.pth")
if MLSD: models_to_download.append("control_sd15_mlsd.pth")
if Normal: models_to_download.append("control_sd15_normal.pth")
if OpenPose: models_to_download.append("control_sd15_openpose.pth")
if Scribble: models_to_download.append("control_sd15_scribble.pth")
if Segmentation: models_to_download.append("control_sd15_seg.pth")

base_url = "https://huggingface.co/lllyasviel/ControlNet/resolve/main/models/"
model_dir = "/content/ControlNet2026/models"
 #/kaggle/working/ControlNet2026/models if running on Kaggle
# Assicuriamoci che la cartella esista
os.makedirs(model_dir, exist_ok=True)

if not models_to_download:
    print("⚠️ Nessun modello selezionato. Spunta almeno una casella.")
else:
    print(f"🚀 Inizio controllo e download di {len(models_to_download)} modelli...\n")

    for model in models_to_download:
        model_path = os.path.join(model_dir, model)
        if not os.path.exists(model_path):
            print(f"⬇️ Scaricando {model}...")
            # Scarica in modo silenzioso mostrando solo la progress bar
            !wget -q --show-progress -O "{model_path}" "{base_url}{model}"
        else:
            print(f"✅ {model} già presente, salto il download.")

    print("\n🎉 Download completato!")

🚀 Inizio controllo e download di 1 modelli...

⬇️ Scaricando control_sd15_canny.pth...
/content/ControlNet 100%[===================>]   5.32G   209MB/s    in 27s     

🎉 Download completato!


In [4]:
import os
import sys

"""
--- PATCH BASICSR vs TORCHVISION (V2) ---
Aggiorna l'import senza tentare di caricare la libreria rotta!
"""

print("Cerco il file difettoso di basicsr...")

# Troviamo la cartella dist-packages senza importare basicsr
dist_packages = [p for p in sys.path if 'dist-packages' in p][0]
degradations_file = os.path.join(dist_packages, 'basicsr', 'data', 'degradations.py')
print(degradations_file)

if os.path.exists(degradations_file):
    with open(degradations_file, 'r', encoding='utf-8') as f:
        content = f.read()

    vecchio_import = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    nuovo_import = "from torchvision.transforms.functional import rgb_to_grayscale"

    if vecchio_import in content:
        content = content.replace(vecchio_import, nuovo_import)

        with open(degradations_file, 'w', encoding='utf-8') as f:
            f.write(content)
        print("✅ Patch applicata con successo! Libreria basicsr svecchiata.")
    else:
        print("⚠️ Patch già applicata o stringa non trovata.")
else:
    print(f"❌ Impossibile trovare il file: {degradations_file}")

Cerco il file difettoso di basicsr...
/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
✅ Patch applicata con successo! Libreria basicsr svecchiata.


In [5]:
import os
import sys
import pytorch_lightning

# Fix per PyTorch Lightning 2.x
pl_path = os.path.dirname(pytorch_lightning.__file__)
dist_file = os.path.join(pl_path, 'utilities', 'distributed.py')

if not os.path.exists(dist_file):
    print("Applicazione patch di compatibilità per PyTorch Lightning...")
    os.makedirs(os.path.dirname(dist_file), exist_ok=True)
    with open(dist_file, 'w') as f:
        f.write('from pytorch_lightning.utilities.rank_zero import *')
    print("✅ Patch applicata.")

Applicazione patch di compatibilità per PyTorch Lightning...
✅ Patch applicata.


In [6]:
%%capture
!pip install transformers

In [19]:
!python word_swap.py

No module 'xformers'. Proceeding without it.
ControlLDM: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
Loading weights: 100% 196/196 [00:00<00:00, 879.03it/s, Materializing param=text_model.final_layer_norm.weight]
CLIPTextModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.l

In [ ]:
!python gradio_swap.py

No module 'xformers'. Proceeding without it.
ControlLDM: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
Loading weights: 100% 196/196 [00:00<00:00, 857.76it/s, Materializing param=text_model.final_layer_norm.weight]
CLIPTextModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.s